# 04 · Agregación y clasificación final

<a href="https://colab.research.google.com/github/manuelarguelles/tyv-demo-colab/blob/main/notebooks/04_agregacion_clasificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Último paso: las **siete notas** (una por criterio de la rúbrica real AL,
del notebook anterior) se combinan en un **subtotal curricular sobre 60
puntos**, y ese subtotal se traduce en una **categoría** para el candidato.

Este notebook se enfoca **solo en la matemática de agregación** — por eso,
a diferencia de `01`–`03`, no pide subir un PDF: usa un **conjunto de 7
niveles de ejemplo** (declarados explícitamente como tal más abajo, no
extraídos de ningún CV) para poder mostrar la fórmula con números
concretos. Para ver estos mismos 7 niveles calculados a partir de **tu**
CV real, corré `00_pipeline_completo.ipynb` o `03_siete_consultas_llm.ipynb`
con tu clave de DeepSeek activa.

**La fórmula (peso por dimensión, no por criterio individual):**

| Dimensión    | Criterios (perfil AL) | Peso   |
|--------------|------------------------|--------|
| Formación    | AL_01, AL_02, AL_03    | 20 %   |
| Experiencia  | AL_04, AL_05           | 25 %   |
| Técnico      | AL_06, AL_07           | 15 %   |

$$\text{Subtotal} = \underbrace{\frac{\sum F}{3\times3}\times20}_{Formación} + \underbrace{\frac{\sum E}{2\times3}\times25}_{Experiencia} + \underbrace{\frac{\sum T}{2\times3}\times15}_{Técnico}$$

**Los umbrales de clasificación** (sobre el subtotal expresado en
porcentaje, 0–100 %):

| Rango          | Categoría          |
|----------------|---------------------|
| < 60 %         | No apto             |
| 60 % – 80 %    | Reserva             |
| ≥ 80 %         | Apto para entrevista |

Esta clasificación es **solo del filtro curricular** — la entrevista
personal (hasta 40 puntos adicionales) queda fuera de este proceso.


## 1. Niveles de ejemplo (ILUSTRATIVOS — no provienen de un CV real)

Usamos estos 7 valores solo para poder mostrar la fórmula corriendo. Reemplazalos por el `resultados` real que te devuelva `03` si querés ver tu propio caso.

In [ ]:
# ⚠ ILUSTRATIVO: valores de ejemplo, no la evaluación de ningún candidato real.
resultados = {
    "AL_01": {"valor": 2, "cita_verificada": True},
    "AL_02": {"valor": 3, "cita_verificada": True},
    "AL_03": {"valor": 2, "cita_verificada": True},
    "AL_04": {"valor": 3, "cita_verificada": True},
    "AL_05": {"valor": 2, "cita_verificada": True},
    "AL_06": {"valor": 2, "cita_verificada": True},
    "AL_07": {"valor": 3, "cita_verificada": True},
}
DIMENSION_DE = {"AL_01": "Formación", "AL_02": "Formación", "AL_03": "Formación",
                 "AL_04": "Experiencia", "AL_05": "Experiencia",
                 "AL_06": "Técnico", "AL_07": "Técnico"}
for cid, r in resultados.items():
    print(f"{cid}  ({DIMENSION_DE[cid]:12s}) → nivel {r['valor']}")


## 2. Agregación: subtotal sobre 60

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

GRUPOS = {
    "Formación":   (["AL_01", "AL_02", "AL_03"], 20),
    "Experiencia": (["AL_04", "AL_05"],          25),
    "Técnico":     (["AL_06", "AL_07"],          15),
}
CLASES = ("no apto", "reserva", "apto entrevista")

def calcular_subtotal(resultados: dict, grupos: dict = GRUPOS) -> dict:
    dimensiones, faltantes = {}, []
    for dimension, (criterios, peso) in grupos.items():
        valores = [resultados[c]["valor"] for c in criterios if resultados.get(c, {}).get("valor") in (1, 2, 3)]
        if len(valores) == len(criterios):
            puntos = (Decimal(sum(valores)) * peso / (3 * len(criterios))).quantize(
                Decimal("0.1"), rounding=ROUND_HALF_UP)
            dimensiones[dimension] = float(puntos)
        else:
            dimensiones[dimension] = None
            faltantes += [c for c in criterios if resultados.get(c, {}).get("valor") not in (1, 2, 3)]
    total = None if faltantes else round(sum(dimensiones.values()), 1)
    return {"total": total, "dimensiones": dimensiones, "faltantes": faltantes}

def clasificar(subtotal, umbral_reserva: float = 60, umbral_apto: float = 80) -> dict:
    if subtotal is None:
        return {"porcentaje": None, "etiqueta": None}
    porcentaje = subtotal * 100 / 60
    etiqueta = CLASES[0] if porcentaje < umbral_reserva else CLASES[1] if porcentaje < umbral_apto else CLASES[2]
    return {"porcentaje": round(porcentaje, 1), "etiqueta": etiqueta}

calculo = calcular_subtotal(resultados)
for dimension, puntos in calculo["dimensiones"].items():
    print(f"{dimension:12s} → {puntos} / {GRUPOS[dimension][1]}")
print(f"\nSubtotal curricular: {calculo['total']} / 60")


## 3. Clasificación final

In [ ]:
clasificacion = clasificar(calculo["total"])
print(f"Puntaje final: {calculo['total']} / 60  →  {clasificacion['porcentaje']}%")
print(f"Categoría: {clasificacion['etiqueta'].upper()}")


## 4. Reporte final (lo que quedaría en el registro auditable)

In [ ]:
informe = {
    "subtotal_curricular": calculo["total"], "maximo": 60,
    "porcentaje": clasificacion["porcentaje"], "categoria": clasificacion["etiqueta"],
    "dimensiones": calculo["dimensiones"],
    "detalle_por_criterio": {cid: {"dimension": DIMENSION_DE[cid], **resultados[cid]} for cid in resultados},
}
import json
print(json.dumps(informe, indent=2, ensure_ascii=False))


## 5. Sensibilidad: cómo cambia la categoría según el desempeño

Perfiles de referencia (no candidatos reales) para entender los umbrales.

In [ ]:
perfiles = {
    "Todo nivel 1 (mínimo)": {cid: 1 for cid in DIMENSION_DE},
    "Todo nivel 2 (cumple)": {cid: 2 for cid in DIMENSION_DE},
    "Todo nivel 3 (máximo)": {cid: 3 for cid in DIMENSION_DE},
    "Nuestro ejemplo (mixto)": {cid: resultados[cid]["valor"] for cid in DIMENSION_DE},
}
print(f"{'Perfil':28s} {'Subtotal':>10s} {'%':>7s}  Categoría")
for nombre, niveles in perfiles.items():
    r = {cid: {"valor": v} for cid, v in niveles.items()}
    c = calcular_subtotal(r)
    cl = clasificar(c["total"])
    print(f"{nombre:28s} {c['total']:>10} {cl['porcentaje']:>6}%  {cl['etiqueta']}")


## Fin del recorrido paso a paso

Este notebook cierra el pipeline de 4 etapas. Para correr el flujo
**completo con tu propio CV real, de punta a punta**, abrí
`00_pipeline_completo.ipynb`.

---
*Este material es contenido educativo de apoyo a una tesis de maestría (Terry & Valdez — sistema de filtrado curricular). El PDF que subís y el nombre que ingresás quedan solo en la memoria de esta sesión de Colab — nunca se guardan en este repositorio ni se envían a ningún lado salvo, si activás el modo real, al proveedor del modelo (DeepSeek), y solo el texto ya anonimizado. Ver `materiales/README.md` para trabajar con archivos reales en disco de forma local.*
